# 01 — Download data

Pull raw legal-source files into `data/raw/<source>/...` so the rest of the pipeline (parse → load → embed → query) has something to work with.

Six ingest verbs ship under `clg ingest`:

| Verb | Jurisdiction | Identifier scheme | Cost / licence |
|---|---|---|---|
| `eurlex` | EU | CELEX (`32016R0679`) | Open, no key |
| `retsinformation` | DK statute | ELI (`lbk/2018/502`) | Open, no key |
| `domstol` | DK case law | ECLI (`ECLI:DK:HR:2023:1234`) + URL | Open, attribution |
| `karnov` | DK commercial reporter | proprietary | **Skeleton** — needs subscription + `KARNOV_API_KEY` |
| `legislation-uk` | UK | slash-form (`ukpga/2006/35`) | Open Government Licence v3.0 |
| `courtlistener` | US | dump date (`2024-12-31`) | Open, polite rate limit |
| `find-case-law` | UK case law (TNA) | — | **Gated** — needs `TNA_COMPUTATIONAL_LICENCE_ACCEPTED=1` |
| `uscode` | US Code USLM | — | Deferred (Phase 1 stub) |

Each verb is **resumable** (re-runs skip already-cached files), **polite** (rate-limited via `crimellm.common.http.get_with_retry`), and **gated on `is_enabled(<jurisdiction>)`** — drop a code from `ENABLED_JURISDICTIONS` and the corresponding ingest verb refuses to run with a clear error.

**This notebook doesn't need Neo4j running** — it just writes files to disk. Notebooks 02–04 cover the database side.

## Prereqs
- `uv sync --extra clg` (pulls `httpx`, `lxml`, `pypdf`, `eyecite`, …)
- `.env` with `ENABLED_JURISDICTIONS=US,EW,UK,EU,DK` (the default — drop a code to skip that jurisdiction's verb)
- No API keys required for the open sources

In [7]:
import os
import subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
print('cwd:', Path.cwd())

def run(cmd: str, *, check: bool = True) -> None:
    '''Echo the command, run it through the shell, raise on non-zero.'''
    print(f'$ {cmd}')
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print('--- stderr ---')
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f'exit {result.returncode}: {cmd}')

cwd: /Users/paolobozzini/Desktop/CrimeLLM


## 1. EU — `clg ingest eurlex`

Direct-by-CELEX downloader against `publications.europa.eu/resource/celex/<CELEX>?language=...&format=...`. SPARQL discovery deferred — operator works from a known CELEX list.

### Parameters

| Flag | Type | Required | Default | Notes |
|---|---|---|---|---|
| `--celex` | CSV | ✅ | — | One or more CELEX ids, comma-separated |
| `--lang` | CSV | optional | `en` | ISO 639-1 codes. Supported: `en, da, de, fr` (CELLAR serves 24; firm-relevant subset). One file cached per `(CELEX, lang)` pair |
| `--fmt` | str | optional | `fmx4` | `fmx4` = FORMEX (universal); `xhtml_akn` = Akoma Ntoso (newer endpoints). Cache filename embeds the format |

### CELEX cheat sheet

CELEX is an 8–10 char id: `<sector><year><type><num>`.

| Sector | Meaning | Common type letters |
|---|---|---|
| `1` | Treaties | (e.g. `12012E/TXT`) |
| `3` | Legislation | `R` regulation · `L` directive · `D` decision · `H` recommendation |
| `6` | Case law | `CJ` court judgment · `CO` court order · `CC` AG opinion |

Examples:
- `32016R0679` — Regulation (EU) 2016/679 (GDPR)
- `32019L0770` — Directive (EU) 2019/770 (Digital Content)
- `32016L0680` — Directive (EU) 2016/680 (Law Enforcement Directive)
- `62012CJ0131` — Google Spain (ECLI:EU:C:2014:317)
- `61991CJ0267` — Keck Mithouard (ECLI:EU:C:1993:905)

### Discovery
- Browse https://eur-lex.europa.eu/ → search by topic → the CELEX is in the URL/metadata
- Firm matter index → existing case file CELEX list
- (Future) `clg ingest eurlex-discover --since YYYY-MM-DD --type directive` — Phase 3.5 deferred

### Examples

In [8]:
# GDPR (EN only)
#run('uv run clg ingest eurlex --celex 32016R0679')

# GDPR + DCD + LED in both EN and DA (firm default for DK lawyers)
run('uv run clg ingest eurlex --celex 32016R0679,32019L0770,32016L0680 --lang en,da')

# Landmark CJEU judgment, Akoma Ntoso format when available
run('uv run clg ingest eurlex --celex 62012CJ0131 --lang en --fmt xhtml_akn', check=False)

$ uv run clg ingest eurlex --celex 32016R0679,32019L0770,32016L0680 --lang en,da
{
  "32016R0679@en.fmx4": "data/raw/eurlex/32016R0679.en.fmx4.xml",
  "32016R0679@da.fmx4": "data/raw/eurlex/32016R0679.da.fmx4.xml",
  "32019L0770@en.fmx4": "data/raw/eurlex/32019L0770.en.fmx4.xml",
  "32019L0770@da.fmx4": "data/raw/eurlex/32019L0770.da.fmx4.xml",
  "32016L0680@en.fmx4": "data/raw/eurlex/32016L0680.en.fmx4.xml",
  "32016L0680@da.fmx4": "data/raw/eurlex/32016L0680.da.fmx4.xml"
}

$ uv run clg ingest eurlex --celex 62012CJ0131 --lang en --fmt xhtml_akn

--- stderr ---
╭───────────────────── Traceback (most recent call last) ──────────────────────╮
│ /Users/paolobozzini/Desktop/CrimeLLM/src/crimellm/clg/cli/ingest.py:364 in   │
│ eurlex                                                                       │
│                                                                              │
│   361 │   │   raise typer.BadParameter("--celex produced no ids")            │
│   362 │   languages = t

## 2. DK legislation — `clg ingest retsinformation`

Uses the official Retsinformation harvest API at https://api.retsinformation.dk/ (see the swagger UI at https://api.retsinformation.dk/index.html). Two modes:

- **`--discover [--since YYYY-MM-DD]`** — GET `https://api.retsinformation.dk/v1/Documents?date=...` returns the daily-delta JSON feed (changed-in-last-10-days). One record per `Document` with the **accession number** (`accessionsnummer`), `documentType.shortName` (`LBK H`, `BEK H`, `CIR1H`, …), `changeDate`, `reasonForChange`, and the direct-XML `href`.
- **`--items <accn>,<accn>,…`** — GET `http://retsinformation.dk/eli/accn/<ACCN>/xml` for each accn; caches the LexDania XML body as `data/raw/retsinformation/<accn>.xml`.

### Parameters

| Flag | Type | Required | Default | Notes |
|---|---|---|---|---|
| `--items` | CSV | one of | — | Accession numbers (e.g. `A20180050229,B20260050805`). Find via `--discover` or by browsing the SPA |
| `--discover` | bool | one of | `False` | Print the daily-delta JSON feed and exit; use with `--since` |
| `--since` | str | optional | today | ISO date for `--discover`. Must be within the last 10 days (API contract) |

### Accession-number cheat sheet

The accn format encodes type + publication date + sequence number:

| Prefix | Document type | Example |
|---|---|---|
| `A` | Lov / Lovbekendtgørelse (LOV / LBK) | `A20230033929` (LBK 339 of 2023) |
| `B` | Bekendtgørelse (BEK) | `B20260050805` |
| `C` | Cirkulære (CIR / CIR1H) | `C20260935909` |

The `documentType.shortName` in the JSON feed disambiguates further (`LBK H` = consolidated act, `BEK H` = executive order, etc.).

### Document types (`shortName` from the JSON feed)

| Code | DA name | EN gloss |
|---|---|---|
| `LBK H` | Lovbekendtgørelse | Consolidated act (post-amendment republication — the *current* citable form) |
| `LOV` | Lov | Act of Parliament (initial enactment) |
| `BEK H` | Bekendtgørelse | Executive order / regulation |
| `CIR1H`, `CIR` | Cirkulære | Administrative circular |
| `VEJ` | Vejledning | Official guidance |

### Discovery workflow

1. Run `--discover` to harvest the last few days.
2. Pick the accns you want by filtering the JSON (e.g. only `LBK H` records).
3. Pass them to `--items` to actually download the XML.


In [6]:
# 1. Discover what changed recently — pick a date in the last 10 days.
run("uv run clg ingest retsinformation --discover --since 2026-06-05 | head -40", check=False)

# 2. Download a specific accession number (Lægemiddelloven, LBK 339 of 2023).
run("uv run clg ingest retsinformation --items A20230033929")

run('uv run clg ingest retsinformation --items A20180050230 ')

# 3. Verify the cached XML.
import json
from pathlib import Path

for f in sorted(Path('data/raw/retsinformation').glob('*.xml')):
    print(f'{f.name}  {f.stat().st_size:>10,} bytes')


$ uv run clg ingest retsinformation --discover --since 2026-06-05 | head -40
[
  {
    "accn": "C20260935909",
    "documentId": "AA015214",
    "type": "CIR1H",
    "changeDate": "2026-06-05",
    "reasonForChange": "DocumentMetadataChanged",
    "href": "http://retsinformation.dk/eli/accn/C20260935909/xml"
  },
  {
    "accn": "B20260050805",
    "documentId": "DI001782",
    "type": "BEK H",
    "changeDate": "2026-06-05",
    "reasonForChange": "NewDocument",
    "href": "http://retsinformation.dk/eli/accn/B20260050805/xml"
  },
  {
    "accn": "C20260956309",
    "documentId": "AA015297",
    "type": "CIR1H",
    "changeDate": "2026-06-05",
    "reasonForChange": "NewDocument",
    "href": "http://retsinformation.dk/eli/accn/C20260956309/xml"
  },
  {
    "accn": "B20250131505",
    "documentId": "DI001621",
    "type": "BEK H",
    "changeDate": "2026-06-05",
    "reasonForChange": "DocumentMetadataChanged",
    "href": "http://retsinformation.dk/eli/accn/B20250131505/xml"
  },
 

## 3. DK case law — `clg ingest domstol`

domstol.dk doesn't publish a clean direct-by-ECLI URL scheme, so the operator pastes an `(ECLI, URL)` list. PDFs cache per ECLI; the parser tries `pypdf` first, then `[ocr]`-extra `ocrmypdf` for scanned older judgments (Phase 14.2).

### Parameters

| Flag | Type | Required | Default | Notes |
|---|---|---|---|---|
| `--items` | CSV | ✅ | — | CSV of `<ECLI>\|<URL>` pairs (use `\|` as the in-pair delimiter) |

### ECLI:DK schema

`ECLI:DK:<court>:<year>:<seq>` — the court code drives the `Court` node mapping (Phase 5 `DK_COURTS`):

| Code | Court | Level | Notes |
|---|---|---|---|
| `HR` | Højesteret | 3 | Supreme Court of Denmark |
| `OLR` | Østre Landsret | 2 | Eastern High Court |
| `VLR` | Vestre Landsret | 2 | Western High Court |
| `BYR` | Byret | 1 | District court (generic — specific bynames possible but not yet hierarchised) |

Unknown codes are lowercased and used as the slug (forward-compatible).

### URL discovery
- Browse https://domstol.dk/ → search → copy the PDF link to the published judgment
- Firm matter index → existing case files
- Per-court archives: https://domstol.dk/hojesteret/, .../oestre-landsret/, .../vestre-landsret/

### Commercial reporter alternative

`clg ingest karnov` is a **skeleton** — refuses to run without `KARNOV_API_KEY` and a confirmed firm subscription. When the subscription lands, the ingester would pull the full Karnov Online / Ufr corpus with editorial summaries + treatment annotations.

In [ ]:
# Single judgment (operator-supplied URL)
# run("uv run clg ingest domstol --items 'ECLI:DK:HR:2023:1234|https://domstol.dk/.../1234.pdf'")

# Multi-judgment bundle
# run(
#     "uv run clg ingest domstol --items '"
#     "ECLI:DK:HR:2023:1234|https://domstol.dk/.../1234.pdf,"
#     "ECLI:DK:OLR:2023:42|https://.../42.pdf,"
#     "ECLI:DK:VLR:2024:7|https://.../7.pdf"
#     "'"
# )

# Karnov gate (will refuse without KARNOV_API_KEY)
run('uv run clg ingest karnov', check=False)

## 4. UK — `clg ingest legislation-uk`

Whole-act CLML XML per statute and per version. Each request fetches the entire act at one version (~50 KB to ~5 MB depending on size).

### Parameters

| Flag | Type | Required | Default | Notes |
|---|---|---|---|---|
| `--statutes` | CSV | optional | `UK_CRIMINAL_ACTS` bundle | Slash-form ids: `<act_type>/<year>/<num>` |
| `--versions` | CSV | optional | `enacted,current` | Version labels — see below |

### Act type prefixes

Pulled directly from legislation.gov.uk URL conventions:

| Prefix | Meaning |
|---|---|
| `ukpga` | UK Public General Act (most common) |
| `ukla` | UK Local Act |
| `uksi` | UK Statutory Instrument |
| `ukcm` | Church Measure |
| `wsi` | Welsh Statutory Instrument |
| `apgb` | Act of Parliament of Great Britain (pre-1801) |

### Version labels

Each act is published at multiple points in time. Pass any combination as CSV:

| Label | Meaning |
|---|---|
| `enacted` | Original royal-assent text (immutable historical) |
| `current` | Latest amended-version (legislation.gov.uk's "current" snapshot) |
| `YYYY-MM-DD` | Point-in-time as-of that ISO date (e.g. `2018-05-25`) |

Other strings → validation error. The point-in-time variants make `clg query "..." --as-of YYYY-MM-DD` work correctly.

### Default `UK_CRIMINAL_ACTS` bundle

When `--statutes` is omitted, the ingester pulls 9 staple UK criminal-law acts:

| Act | Slash id |
|---|---|
| Fraud Act 2006 | `ukpga/2006/35` |
| Theft Act 1968 | `ukpga/1968/60` |
| Misuse of Drugs Act 1971 | `ukpga/1971/38` |
| Criminal Damage Act 1971 | `ukpga/1971/48` |
| Offences against the Person Act 1861 | `ukpga/1861/100` |
| Public Order Act 1986 | `ukpga/1986/64` |
| Sexual Offences Act 2003 | `ukpga/2003/42` |
| Modern Slavery Act 2015 | `ukpga/2015/30` |
| Bribery Act 2010 | `ukpga/2010/23` |

### Discovery
- Browse https://www.legislation.gov.uk/ → search → the URL has `/<act_type>/<year>/<num>/`
- Lists of UK acts by year: https://www.legislation.gov.uk/browse/year/all

### Examples

In [ ]:
# Default bundle — 9 criminal-law staples, enacted + current
run('uv run clg ingest legislation-uk')

# Single act, two versions
run('uv run clg ingest legislation-uk --statutes ukpga/2006/35 --versions enacted,current')

# Point-in-time slice for as-of queries
run(
    'uv run clg ingest legislation-uk '
    '--statutes ukpga/2006/35 '
    '--versions 2010-01-01,2018-05-25,2023-12-31'
)

## 5. US — `clg ingest courtlistener` (heavy)

Free Law Project publishes monthly + quarterly bulk dumps of the entire CourtListener corpus as bz2 CSV. Each dump is identified by its publication date (`YYYY-MM-DD`). One file per logical table; together they describe ~7M opinions + the citation graph.

### Parameters

| Flag | Type | Required | Default | Notes |
|---|---|---|---|---|
| `--date` | str | ✅ | — | Dump date stamp `YYYY-MM-DD` matching the FLP publication |
| `--files` | CSV | optional | `courts,dockets,clusters,opinions,citations` | Which logical tables to download |
| `--dest` | Path | optional | `data/raw/courtlistener/` | Override the cache dir |

### File keys + sizes (typical, growing over time)

| Key | URL pattern | Approx size (bz2) | Role |
|---|---|---|---|
| `courts` | `courts-<date>.csv.bz2` | ~1 MB | Court hierarchy / metadata |
| `dockets` | `dockets-<date>.csv.bz2` | ~2 GB | One row per docket |
| `clusters` | `opinion-clusters-<date>.csv.bz2` | ~500 MB | One row per opinion-cluster (≈ one judgment) |
| `opinions` | `opinions-<date>.csv.bz2` | ~30 GB | Per-opinion text bodies (the heavy file) |
| `citations` | `citation-map-<date>.csv.bz2` (fallbacks: `opinionscited-…`, `opinions-cited-…`) | ~500 MB | Opinion→opinion citation graph → drives `CITES` edges |
| `reporter_citations` | `citations-<date>.csv.bz2` | ~200 MB | Reporter cites (`467 U.S. 837`) → `Case.citations[]` |

### Date discovery
- Browse https://storage.courtlistener.com/bulk-data/ → latest dump dates
- Recent dumps typically end-of-month
- Each `--date` must match a real published dump or you'll get 404s

### Status + sidecar build

Two helper verbs make the slow `opinions` file manageable:

```
clg ingest courtlistener-status --date YYYY-MM-DD     # which files are cached?
clg ingest courtlistener-index  --date YYYY-MM-DD     # build the opinion→cluster sidecar (one-time, slow)
```

The sidecar is a slim `opinion_id,cluster_id` index built by a single pass over `opinions.csv.bz2`. Without it, `clg load courtlistener` would re-stream the 30 GB file every time it needed to resolve a citation edge. With it, the load reads only the sidecar (~50 MB).

### Examples

In [ ]:
# DATE = '2024-12-31'   # ← look this up from FLP's bulk index first

# Heavy — uncomment when US is in scope and the date is verified.
# run(f'uv run clg ingest courtlistener --date {DATE} --files courts,clusters,citations')
# run(f'uv run clg ingest courtlistener-status --date {DATE}')
# run(f'uv run clg ingest courtlistener-index  --date {DATE}')   # ~1-2 hr on the full opinions.bz2
print('skipped — uncomment when US is in scope')

## 6. UK case law — `clg ingest find-case-law` (gated)

The National Archives publishes UK High Court / Court of Appeal / Supreme Court judgments at caselaw.nationalarchives.gov.uk. **Programmatic bulk use requires a free computational-analysis licence** — see https://caselaw.nationalarchives.gov.uk/computational-licence.

Once you've applied + been approved, set `TNA_COMPUTATIONAL_LICENCE_ACCEPTED=1` in `.env` and the verb unlocks. Rate limit ~1000 req / 5 min — the ingester is polite by default.

In [ ]:
run('uv run clg ingest find-case-law', check=False)

## 7. US Code — `clg ingest uscode`

Phase 1 stub. Defer until USLM bulk parsing is wired in.

## Inspect what landed on disk

Per-source counts + total size, plus a peek at the first few filenames.

In [ ]:
raw = Path('data/raw')
if not raw.exists():
    print("data/raw doesn't exist yet — run an ingest verb first.")
else:
    for source_dir in sorted(raw.iterdir()):
        if not source_dir.is_dir():
            continue
        files = sorted(source_dir.glob('*'))
        total_mb = sum(f.stat().st_size for f in files if f.is_file()) / 1e6
        print(f'{source_dir.name:<24} {len(files):>4} files   {total_mb:>8.1f} MB')
        for f in files[:3]:
            print(f'    {f.name}')
        if len(files) > 3:
            print(f'    … ({len(files) - 3} more)')

## Cache layout reference

| Source | Path pattern |
|---|---|
| `eurlex` | `data/raw/eurlex/<CELEX>.<lang>.<fmt>.xml` |
| `retsinformation` | `data/raw/retsinformation/<doc_type>-<year>-<num>.xml` |
| `domstol` | `data/raw/domstol/ECLI_DK_<COURT>_<year>_<seq>.pdf` (or `.html`) |
| `legislation_uk` | `data/raw/legislation_uk/<act_type>-<year>-<num>-<version>.xml` |
| `courtlistener` | `data/raw/courtlistener/<table>-<date>.csv.bz2` |

All paths are settable via `Settings.raw_root` (default `data/raw/`) and per-source overrides where applicable.

## Fixtures alternative (offline development)

All five jurisdictions ship synthetic / redacted fixtures under `tests/clg/fixtures/` — useful when network is unreliable or for fast iteration. Notebooks 02–04 default to fixtures so the full pipeline runs without any downloads.

In [ ]:
fix = Path('tests/clg/fixtures')
for d in sorted(fix.iterdir()):
    if d.is_dir():
        files = sorted(d.glob('*'))
        print(f'{d.name}/  ({len(files)} fixtures)')
        for f in files:
            print(f'    {f.name}')

## Removability invariant (Phase 11)

Each ingest verb is **gated on `is_enabled(<jurisdiction>)`**. Drop a code from `ENABLED_JURISDICTIONS` and the verb refuses:

```bash
ENABLED_JURISDICTIONS=DK,EU clg ingest courtlistener --date ...    # → BadParameter: 'US' not in [DK, EU]
ENABLED_JURISDICTIONS=US,UK clg ingest retsinformation --items ... # → BadParameter: 'DK' not in [US, UK]
```

Re-enable any time and re-run — already-downloaded files are preserved and the verb is idempotent (resumable cache).

## Next: notebook 02 — set up the database

Once raw files are on disk (or you've decided to use fixtures), [`02_setup_database.ipynb`](02_setup_database.ipynb) spins up Neo4j, applies the schema, and demonstrates the removability invariant via `ENABLED_JURISDICTIONS`.